In [ ]:
import minari
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms


class MinariDataset(Dataset):
    def __init__(self, dataset_name):
        print(f"Loading Minari dataset '{dataset_name}'...")
        dataset = minari.load_dataset(dataset_name)
        self.observations = np.vstack(
            [e.observations for e in dataset.iterate_episodes()]
        )

        # The ToTensor transform handles scaling to [0,1] and dimension permutation (H,W,C -> C,H,W)
        self.transform = transforms.ToTensor()
        print("Dataset loaded and observations extracted.")

    def __len__(self):
        return len(self.observations)

    def __getitem__(self, idx):
        obs = self.observations[idx]
        return self.transform(obs)


full_dataset = MinariDataset("Box2D/CarRacing-v3/expert-v0")
print(f"Total observations: {len(full_dataset)}")
print(f"Observation shape: {full_dataset[0].shape}")

# train test split
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

batch_size = 64

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

print("\nData preparation complete.")
print(f"Train DataLoader ready with {len(train_loader)} batches of size {batch_size}.")
print(
    f"Validation DataLoader ready with {len(val_loader)} batches of size {batch_size}."
)

Loading Minari dataset 'Box2D/CarRacing-v3/expert-v0'...
Dataset loaded and observations extracted.
Total observations: 245743
Observation shape: torch.Size([3, 96, 96])
Training set size: 221168
Validation set size: 24575

Data preparation complete.
Train DataLoader ready with 3456 batches of size 64.
Validation DataLoader ready with 384 batches of size 64.


In [ ]:
from tqdm import tqdm
from src.models.vae import VAE, vae_loss

# Hyperparameters
epochs = 20
learning_rate = 1e-3
latent_dim = 32
beta = 1.0 # Weight for the KL divergence term

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VAE(latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(f"Training on {device}...")

# Training Loop
for epoch in range(epochs):
    model.train()
    train_loss = 0
    
    # Use tqdm for a nice progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch in pbar:
        # Data is a list with one element, the image tensor
        data = batch[0].to(device)
        
        # Forward pass
        recon_batch, mu, log_var = model(data)
        
        # Calculate loss
        loss = vae_loss(recon_batch, data, mu, log_var, beta)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': loss.item() / len(data)})

    avg_loss = train_loss / len(train_loader.dataset)
    print(f"====> Epoch: {epoch+1} Average loss: {avg_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), 'carracing_vae.pth')
print("Model training complete and saved to carracing_vae.pth")

In [ ]:
# 1. Load your trained model
model = VAE(latent_dim=latent_dim)
model.load_state_dict(torch.load('carracing_vae.pth'))
model.to(device)
model.eval() # Set the model to evaluation mode

# 2. Get a single observation and preprocess it
sample_obs = all_obs[42] # Get any observation
image = torch.from_numpy(sample_obs.astype(np.float32) / 255.0)
image = image.permute(2, 0, 1).unsqueeze(0).to(device) # (H,W,C) -> (C,H,W) -> (1,C,H,W)

# 3. Encode the image
with torch.no_grad(): # No need to calculate gradients
    mu, log_var = model.encode(image)

# The mean 'mu' is typically used as the final compressed representation
encoded_vector = mu.cpu().numpy().flatten()

print(f"Original image shape: {sample_obs.shape}")
print(f"Encoded vector shape: {encoded_vector.shape}")
print(f"Encoded vector (mu): \n{encoded_vector}")